In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

* Is cell me tumne project ke liye zaruri libraries import ki hain. Pandas aur NumPy data handle karne ke liye, Matplotlib aur Seaborn visualization ke liye, aur Scikit-Learn machine learning model banane ke liye use kiya hai. Yahi tools poore project me kaam aayenge.

In [ ]:
df = pd.read_csv("Message_Intelligence_Dataset_5200 .csv")
df.head()

* Yahan tumne CSV file ko DataFrame me load kiya aur head() se pehli 5 rows dekhi. Isse hume dataset ka structure aur columns samajhne me madad milti hai.

PART B : Exploratory Data Analysis (EDA)

* Ye ek heading hai jo batati hai ki ab Exploratory Data Analysis (EDA) shuru ho raha hai. EDA ka matlab data ko samajhna aur uski quality check karna hota hai.

In [ ]:
df.shape

* df.shape se tumne check kiya ki dataset me kitni rows aur kitne columns hain. Isse data ka size pata chalta hai.

In [ ]:
df.info()

* df.info() se har column ka datatype, non-null values aur memory usage dekha. Isse pata chalta hai ki kaunsa column numeric hai aur kaunsa text hai.

In [ ]:
df.isnull().sum()

* df.isnull().sum() se tumne har column me missing values check ki hain. Machine Learning model lagane se pehle missing values dhoondhna bahut zaruri hota hai.

In [ ]:
df = df.dropna()

* dropna() use karke tumne missing values wali rows hata di. Isse dataset clean ho gaya aur model training me problem aane ke chances kam ho gaye.

In [ ]:
# Calculate Correlation Matrix

correlation_matrix = df.corr(numeric_only=True)

correlation_matrix

* Yahan numeric columns ke beech relation check kiya gaya hai. Correlation batata hai ki do variables ek dusre ko kitna affect karte hain.

In [ ]:
# Heatmap

# Plot Heatmap

plt.figure(figsize=(14,10))

sns.heatmap(

    correlation_matrix,

    annot=True,

    cmap="coolwarm",

    fmt=".2f",

    linewidths=0.5

)

plt.title("Correlation Heatmap")

plt.show()

* Is cell me correlation matrix ko heatmap ke form me visualize kiya. Heatmap ki wajah se strong aur weak relationships easily samajh me aa jati hain.

In [ ]:
# Drop ID column

df.drop(columns=["message_id"], inplace=True)

df.head()

* message_id sirf ek unique identifier hai aur prediction me koi help nahi karta. Isliye tumne ise dataset se remove kar diya.

In [ ]:
# Check Remaining Columns

df.columns

* ID column delete hone ke baad tumne check kiya ki dataset me ab kaun-kaun se columns bache hain. Ye verification step hai.

Part C: K-Nearest Neighbors

* Ye heading batati hai ki ab tum K-Nearest Neighbors (KNN) algorithm implement karne wale ho.

In [ ]:
# Convert spam and ham into numbers

label_encoder = LabelEncoder()

df["spam_label"] = label_encoder.fit_transform(df["spam_label"])

* Machine Learning model text values (spam, ham) ko directly nahi samajhta. Isliye LabelEncoder ki help se unhe numbers me convert kiya gaya.

In [ ]:
# Convert Message Text into Numeric Features

from sklearn.feature_extraction.text import TfidfVectorizer

# Convert message text into TF-IDF vectors

tfidf = TfidfVectorizer(stop_words="english", max_features=3000)

text_features = tfidf.fit_transform(df["message_text"])

* Message text ko model directly samajh nahi sakta. TF-IDF Vectorizer text ko numbers me convert karta hai aur important words ko zyada importance deta hai.

In [ ]:
# Convert TF-IDF Matrix into DataFrame

text_df = pd.DataFrame(
    text_features.toarray(),
    columns=tfidf.get_feature_names_out()
)

* TF-IDF se jo matrix bani thi usko DataFrame me convert kiya gaya taaki data ko easily dekh aur process kiya ja sake. Ab har important word ek feature (column) ban gaya hai.

In [ ]:
# Select Numerical Features

numerical_features = [

    "message_length",

    "word_count",

    "num_urls",

    "num_digits",

    "num_special_chars",

    "spam_keyword_score",

    "legit_keyword_score",

    "sender_activity_score",

    "sender_account_age_days",

    "messages_sent_last_24h",

    "hour_of_day",

    "day_of_week"

]

* Is cell me tumne sirf un columns ko select kiya hai jo numbers me hain, jaise message length, word count, URLs, digits, special characters etc.

In [ ]:
# Combine Text and Numeric Features

x = pd.concat([

    text_df,

    df[numerical_features].reset_index(drop=True)

], axis=1)

y = df["spam_label"]

* Yahan tumne TF-IDF text features aur numeric features ko ek hi dataset me combine kiya. Saath hi target variable (spam_label) ko y me store kiya.

In [ ]:
# Split Dataset

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraning Shape:", x_train.shape)
print("Testing Shape: ", x_test.shape)

* Is cell me dataset ko 80% training aur 20% testing me divide kiya. stratify=y use kiya taaki dono datasets me spam aur ham ka ratio same rahe.

In [ ]:
# Feature Scaling

scaler = StandardScaler(with_mean=False)

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

* Yahan StandardScaler use karke features ko scale kiya.

In [ ]:
# Train KNN Model

knn = KNeighborsClassifier(
    
    n_neighbors=5
)

knn.fit(
    
    x_train,
    
    y_train
    
)

* Is cell me KNeighborsClassifier banaya aur n_neighbors=5 rakha. Phir model ko training data se train kiya.

In [ ]:
y_pred = knn.predict(x_test)

* Yahan trained KNN model se test data ki prediction nikali.

In [ ]:
# Accuracy

Accuracy = accuracy_score (
    
    y_test,
    
    y_pred
)

print("\nAccuracy: ", Accuracy)

* Is cell me accuracy_score calculate kiya.

In [ ]:
# Confusion Matrix

cm = confusion_matrix (
    
    y_test,
    
    y_pred
)

print("\nConfusion Matrix :", cm)

* Yahan confusion matrix nikala.

In [ ]:
print(classification_report(y_test, y_pred))

* Is cell me Precision, Recall, F1-Score aur Support dekha.

In [ ]:
accuracy_list = []

k_value = range(1,10)

for k in k_value:
    
    model = KNeighborsClassifier (
        
        n_neighbors=k
        
    )
    
    model.fit(
        
        x_train,
        
        y_train
    )
    
    prediction = model.predict(x_test)
    
    score = accuracy_score(y_test,prediction)
    
    accuracy_list.append(score)
    
    print("Accuracy List :")
    print(accuracy_list)

* Yahan K=1 se K=9 tak model train kiya aur har K ki accuracy list me store ki.

In [ ]:
plt.Figure(figsize=(10,5))

plt.plot(
    k_value,
    accuracy_list,
    marker="o"
)

plt.xlabel("K Value")
plt.ylabel("Accuracy")
plt.title("Accuracy for Different K Values")
plt.grid(True)
plt.show()

* Is cell me different K values ki accuracy ka graph banaya.

In [ ]:
# Find Best K

best_k = accuracy_list.index(max(accuracy_list)) + 1

print("Best K :", best_k)

print("Best Accuracy :", max(accuracy_list))

* Yahan maximum accuracy wali K value nikali.

In [ ]:
# Identify Misclassified Messages

misclassified = x_test[y_test != y_pred]

print("Total Misclassified Messages :", len(misclassified))

* Is cell me un messages ko identify kiya jo model ne galat predict kiye.

Part D: Support Vector Machine (SVM)

* Ye markdown cell hai jo batata hai ki ab Support Vector Machine (SVM) model ka implementation shuru ho raha hai.

In [ ]:
from sklearn.svm import SVC

* Is cell me SVC import kiya.

In [ ]:
# Create SVM model with Linear Kernel

svm_liner = SVC(
    
    kernel='linear',
    
    random_state=42
)

* Is cell me tumne Linear Kernel ke saath SVM model banaya.

In [ ]:
# # Train the model

svm_liner.fit(x_train, y_train)

* Is cell me Linear SVM ko training data se train kiya.

In [ ]:
# Predict test data

linear_predicton = svm_liner.predict(x_test)

* Yahan trained model se test data ki prediction nikali.

In [ ]:
# Accuracy

linear_accuary = accuracy_score(
    y_test,
    linear_predicton
)

print("Linear SVM Accuarcy :", linear_accuary)   

* Is cell me accuracy_score calculate kiya.

In [ ]:
# Confusion Matrix

linear_cm = confusion_matrix(
    
    y_test,
    
    linear_predicton
)

print(linear_cm)

* Yahan Confusion Matrix banayi.

In [ ]:
# Classification Report

print(classification_report(y_test,linear_predicton))

* Is cell me Precision, Recall aur F1-Score print kiya.

In [ ]:
# Train SVM with RBF Kernel

svm_rbf = SVC(
    kernel='rbf',
    random_state=42
)

* Yahan tumne RBF (Radial Basis Function) Kernel ke saath ek naya SVM model banaya.

In [ ]:
# Train Model

svm_rbf.fit(x_train,y_train)

* Is cell me RBF SVM ko training data par train kiya.

In [ ]:
# Prediction

rbf_prediction = svm_rbf.predict(x_test)

* Yahan RBF model se test data ki prediction nikali.

In [ ]:
# Accuracy

rbf_accuracy = accuracy_score(y_test, rbf_prediction)

print("RBF SVM Accuracy :", rbf_accuracy)

* Is cell me RBF SVM ki accuracy calculate ki aur print ki.

In [ ]:
# Confusion Matrix

rbf_cm = confusion_matrix(y_test, rbf_prediction)

print(rbf_cm)

* Yahan RBF model ki confusion matrix print ki.

In [ ]:
# Classification Report

print(classification_report(y_test, rbf_prediction))

* Is cell me RBF model ka Precision, Recall aur F1-Score print kiya.

In [ ]:
# Total Support Vectors

print("Total Support Vectors :")

print(

    svm_rbf.n_support_

)

* Yahan model ke Support Vectors print kiye.

In [ ]:
# Total Number of Support Vectors

print("Total Support Vectors Used :", len(svm_rbf.support_))

* Is cell me total support vectors ki count print ki.

In [ ]:
# First 10 Support Vector Indexes

print(svm_rbf.support_[:10])

* Yahan first 10 support vectors ke indexes print kiye.

In [ ]:
# Support Vector Percentage

support_percentage = (

    len(svm_rbf.support_) /

    len(x_train)

) * 100

print(

    "Support Vector Percentage :",

    round(support_percentage,2),

    "%"

)

* Is cell me tumne calculate kiya ki training data me se kitne percent data points Support Vectors bane.

In [ ]:
# Compare Accuracy

print("KNN Accuracy :", Accuracy)

print("Linear SVM Accuracy :", linear_accuary)

print("RBF SVM Accuracy :", rbf_accuracy)

* Is cell me KNN, Linear SVM aur RBF SVM ki accuracy ek saath print ki.

In [ ]:
# Accuracy Comparison Graph

models = [

    "KNN",

    "Linear SVM",

    "RBF SVM"

]

scores = [

    Accuracy,

    linear_accuary,

    rbf_accuracy

]

plt.figure(figsize=(8,5))

plt.bar(

    models,

    scores

)

plt.xlabel("Models")

plt.ylabel("Accuracy")

plt.title("KNN vs SVM Accuracy Comparison")

plt.show()

* Yahan bar graph banaya jisme har model ki accuracy dikhayi.

In [ ]:
# Best SVM Model

if linear_accuary > rbf_accuracy:

    print("Linear Kernel performed better.")

elif linear_accuary < rbf_accuracy:

    print("RBF Kernel performed better.")

else:

    print("Both kernels performed equally.")

* Is cell me condition laga kar check kiya ki Linear SVM aur RBF SVM me se kaunsa better perform kar raha hai.

Part E: Naive Bayes Classifier & Probability

* Ye markdown cell hai jo batata hai ki ab Naive Bayes Classifier ka implementation start ho raha hai.

In [ ]:
from sklearn.naive_bayes import GaussianNB

* Is cell me GaussianNB import kiya.

In [ ]:
x_train_dense = x_train

x_test_dense = x_test

print("\nTrain_dense :", x_train_dense)
print("Test_dense :", x_test_dense)

* Yahan training aur testing data ko print karke verify kiya.

In [ ]:
naive_bayes = GaussianNB()

* Is cell me GaussianNB() object banaya.

In [ ]:
# Train the Model

naive_bayes.fit(x_train_dense, y_train)

* Is cell me Naive Bayes model ko training data se fit kiya.

In [ ]:
# Predict Test Data

nb_prediction = naive_bayes.predict(

    x_test

)

* Yahan trained model se test data ki prediction nikali.

In [ ]:
# Calculate Accuracy

nb_accuracy = accuracy_score(y_test, nb_prediction)

print("Naive Bayes Accuracy :", nb_accuracy)

* Is cell me Naive Bayes model ki accuracy calculate ki.

In [ ]:
# Confusion Matrix

nb_cm = confusion_matrix(y_test, nb_prediction)

print(nb_cm)

* Yahan Confusion Matrix print ki.

In [ ]:
# Classification Report

print(

    classification_report(

        y_test,

        nb_prediction

    )

)

* Is cell me Precision, Recall aur F1-Score print kiya.

In [ ]:
# Train SVM With RBF Kernel

svm_rbf = SVC(
    kernel='rbf',
    random_state=42
)

* Yahan fir se SVC(kernel='rbf') model create kiya.

In [ ]:
# Train Model

svm_rbf.fit(x_train,y_train)

* Is cell me RBF SVM ko training data par fit kiya.

In [ ]:
# Prediction

rbf_prediction = svm_rbf.predict(x_test)

* Yahan RBF SVM model se test data ki prediction nikali.

In [ ]:
# Accuracy

rbf_accuracy = accuracy_score(y_test, rbf_prediction)

print("RBF SVM Accuracy :", rbf_accuracy)

* Is cell me accuracy_score calculate kiya.

In [ ]:
# Confusion Matrix

rbf_cm = confusion_matrix(y_test, rbf_prediction)

print(rbf_cm)

* Yahan Confusion Matrix banayi.

In [ ]:
# Classification Report

print(classification_report(y_test, rbf_prediction))

* Is cell me Precision, Recall aur F1-Score print kiya.

In [ ]:
# Total Support Vectors

print("Total Support Vectors :")
print(svm_rbf.n_support_)

* Yahan model ke sabhi Support Vectors print kiye.

In [ ]:
# Total Number of Support Vectors

print("Total Support Vectors Used :", len(svm_rbf.support_))

* Is cell me total support vectors ki count print ki.

In [ ]:
# First 10 Support Vector Indexes

print(svm_rbf.support_[:10])

* Yahan pehle 10 support vectors ke indexes dekhe.

In [ ]:
# Support Vector Percentage

support_percentage = (

    len(svm_rbf.support_) /

    len(x_train)

) * 100

print(

    "Support Vector Percentage :",

    round(support_percentage,2),

    "%"

)

* Is cell me calculate kiya ki total training data me se kitna percentage Support Vectors ka hai.

In [ ]:
# Compare Accuracy

print("KNN Accuracy :", Accuracy)
print("Linear SVM Accuracy :", linear_accuary)
print("RBF SVM Accuracy :", rbf_accuracy)

* Yahan KNN, Linear SVM aur RBF SVM ki accuracy ek saath print ki.

In [ ]:
# Accuracy Comparison Graph

models = [

    "KNN",

    "Linear SVM",

    "RBF SVM"

]

scores = [

    Accuracy,

    linear_accuary,

    rbf_accuracy

]

plt.figure(figsize=(8,5))
plt.bar(

    models,

    scores

)

plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.title("KNN vs SVM Accuracy Comparison")
plt.show()

* Is cell me models ki accuracy ka graph banaya.

In [ ]:
# Best SVM Model

if linear_accuary > rbf_accuracy:
    print("Linear Kernel performed better.")
elif linear_accuary < rbf_accuracy:
    print("RBF Kernel performed better.")
else:
    print("Both kernels performed equally.")

* Yahan condition laga kar Linear SVM aur RBF SVM me se better model choose kiya.

In [ ]:
# Concusion

print("----- Model Summary -----")

print(f"KNN Accuracy        : {Accuracy:.4f}")

print(f"Linear SVM Accuracy : {linear_accuary:.4f}")

print(f"RBF SVM Accuracy    : {rbf_accuracy:.4f}")

* Is cell me sabhi models ka final summary print kiya.

Part E : Naive Bayes Classifier & Probability

* Ye markdown heading hai jo batati hai ki ab Naive Bayes Classifier & Probability section shuru ho raha hai.

In [ ]:
from sklearn.naive_bayes import GaussianNB

* Yahan GaussianNB library import ki.

In [ ]:
x_train_dense = x_train
x_test_dense = x_test

print("\nTrain_dense :", x_train_dense)
print("Test_dense :", x_test_dense)

* Is cell me x_train aur x_test ko Naive Bayes ke liye prepare kiya.

In [ ]:
naive_bayes = GaussianNB()

* Yahan GaussianNB() ka object banaya.

In [ ]:
# Train the model

naive_bayes.fit(x_train_dense, y_train)

* Model ko fit() karke training data se train kiya.

In [ ]:
# Predict Test Data

nb_prediction = naive_bayes.predict(x_test_dense)

* Test data par prediction ki.

In [ ]:
# Calulate Accuracy

nb_accuracy = accuracy_score(y_test, nb_prediction)

print("Naive Bayes Accuracy :", nb_accuracy)

* Naive Bayes ki accuracy calculate ki.

In [ ]:
# Confusion Matrix

nb_cm = confusion_matrix(y_test, nb_prediction)

print(nb_cm)

* Confusion Matrix print ki.

In [ ]:
# Classification Report

print(classification_report(y_test, rbf_prediction))

* Precision, Recall aur F1-Score print kiya.

In [ ]:
# Class Distribution

print(df["spam_label"].value_counts())

* Spam aur Ham messages ki total count dekhi.

In [ ]:
# Prior Probability

prior_probabilities = df["spam_label"].value_counts(normalize=True)

print(prior_probabilities)

* Har class ki prior probability calculate ki.

In [ ]:
# Display Prior Probabilities 

print("Probabilities of Hum :", prior_probabilities[0])
print("Probabilities of Spam :", prior_probabilities[1])

* Prior probabilities print ki.

In [ ]:
# Predict Class Probabilities

Probabilities = naive_bayes.predict_proba(x_test_dense)

* predict_proba() se har prediction ki probability nikali.

In [ ]:
# Disply First 5 Probabilities

Probabilities[:5]

* Pehli 5 probability values dekhi.

In [ ]:
# Disply Predictded Probabilitie In Table

Probabilities_df = pd.DataFrame(
    Probabilities,
    columns=["Prob_Ham", "Prob_Spam"]
)

Probabilities_df.head()

* Probabilities ko DataFrame me convert kiya.

In [ ]:
# Comapre Actual and Predicted

Comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": nb_prediction})

Comparison_df.head()

* Actual aur Predicted values compare ki.

In [ ]:
# Compare Actual, Prediction And Probability

result = Comparison_df.copy()

result["Ham Probability"] = Probabilities[:,0]
result["Spam Probability"] = Probabilities[:,1]
result.head(10)

* Prediction aur probability ko ek hi table me dikhaya.

In [ ]:
# Predict A Single Message 

Single_Prediction = naive_bayes.predict(x_test_dense[:1])
print(Single_Prediction)

* Ek single message ka prediction kiya.

In [ ]:
# Probability Of Single Message 

Single_Probability = naive_bayes.predict_proba(x_test_dense[:1])
print(Single_Probability)

* Us message ki probability nikali.

In [ ]:
# Disply Result In Simple Formet 

print("Hum Probability :", Single_Probability[0][0])
print("Spam Probability :", Single_Probability[0][1])

* Probability ko simple format me print kiya.

In [ ]:
# Final Comparision

print("===== Naive Bayes Summary =====")

print("Accuracy :", nb_accuracy)

print()

print("Model Prediction :", Single_Prediction[0])

print()

print("Ham Probability :", round(Single_Probability[0][0],4))

print("Spam Probability :", round(Single_Probability[0][1],4))

* Naive Bayes ka final summary print kiya.

In [ ]:
# Simple Conclusion

if Single_Prediction[0] == 1:
    print("The Model Predicted This Message Is Spam")
else:
    print("The Modle Predicted This Message Is Hum")

* Prediction ke basis par final message print kiya.

Part F: Model Compresion Evalution

* Ye markdown heading hai.

In [ ]:
# Calculate KNN Metrics

knn_precision = precision_score(y_test, y_pred)
knn_recall = recall_score(y_test, y_pred)
knn_f1 = f1_score(y_test, y_pred)

print("Knn_Precision :", knn_precision)
print("knn_Recall :", knn_recall)
print("knn_f1 :", knn_f1)

* KNN ka Precision, Recall aur F1 calculate kiya.

In [ ]:
# Calculate Linear SVM Metrics 

linear_Precision = precision_score(y_test, linear_predicton)
linear_recall = recall_score(y_test, linear_predicton)
linear_f1 = f1_score(y_test, linear_predicton)

print("linear_precision :", linear_Precision)
print("linear_recall :", linear_recall)
print("linera_f1 :", linear_f1)

* Linear SVM ke metrics calculate kiye.

In [ ]:
# Calcalute RBF SVM Metrics

rbf_precision = precision_score(y_test, rbf_prediction)
rbf_recall = recall_score(y_test, rbf_prediction)
rbf_f1 = f1_score(y_test, rbf_prediction)

print("rbf_precision :", rbf_precision)
print("rbf_recall :", rbf_recall)
print("rbf_f1 :", rbf_f1)

* RBF SVM ke metrics calculate kiye.

In [ ]:
# Calcalute Navie Bayes Metrics

nb_precision = precision_score(y_test, nb_prediction)
nb_recall = recall_score(y_test, nb_prediction)
nb_f1 = f1_score(y_test, nb_prediction)

print("nb_precision :", nb_precision)
print("nb_recall :", nb_recall)
print("nb_f1 :", nb_f1)

* Naive Bayes ke metrics calculate kiye.

In [ ]:
# Create Comperison Table

comparison_table = pd.DataFrame({

    "Model":[

        "KNN",

        "Linear SVM",

        "RBF SVM",

        "Naive Bayes"

    ],

    "Accuracy":[

        Accuracy,

        linear_accuary,

        rbf_accuracy,

        nb_accuracy

    ],

    "Precision":[

        knn_precision,

        linear_Precision,

        rbf_precision,

        nb_precision

    ],

    "Recall":[

        knn_recall,

        linear_recall,

        rbf_recall,

        nb_recall

    ],

    "F1 Score":[

        knn_f1,

        linear_f1,

        rbf_f1,

        nb_f1

    ]

})

comparison_table

* Sabhi models ki comparison table banayi.

In [ ]:
# Round Values 

comparison_table = comparison_table.round(4)

comparison_table

* Values ko round off kiya.

In [ ]:
# Find Best Accuracy

best_accuracy = comparison_table.loc[

    comparison_table["Accuracy"].idxmax()

]

print(best_accuracy)

* Highest accuracy wala model nikala.

In [ ]:
# Find Best Precison

best_precision = comparison_table.loc[

    comparison_table["Precision"].idxmax()

]

print(best_precision)

* Highest precision wala model nikala.

In [ ]:
# Find Best Recall

best_recall = comparison_table.loc[

    comparison_table["Recall"].idxmax()

]

print(best_recall)

* Highest recall wala model nikala.

In [ ]:
# Find Best F1 Score

best_f1 = comparison_table.loc[

    comparison_table["F1 Score"].idxmax()

]

print(best_f1)

* Highest F1 Score wala model nikala.

In [ ]:
# Print Best Models

print("Best Accuracy Model :", best_accuracy["Model"])

print("Best Precision Model :", best_precision["Model"])

print("Best Recall Model :", best_recall["Model"])

print("Best F1 Score Model :", best_f1["Model"])

* Sabhi best models print kiye.

In [ ]:
# Accurccy Coperison Graph

plt.figure(figsize=(8,5))

plt.bar(

    comparison_table["Model"],

    comparison_table["Accuracy"]

)

plt.title("Accuracy Comparison")

plt.xlabel("Models")

plt.ylabel("Accuracy")

plt.xticks(rotation=10)

plt.show()

* Accuracy comparison graph banaya.

In [ ]:
# Precision Comperison Graph

plt.figure(figsize=(8,5))

plt.bar(

    comparison_table["Model"],

    comparison_table["Precision"]

)

plt.title("Precision Comparison")

plt.xlabel("Models")

plt.ylabel("Precision")

plt.xticks(rotation=10)

plt.show()

* Precision comparison graph banaya.

In [ ]:
# Recall Comparison Graph

plt.figure(figsize=(8,5))

plt.bar(

    comparison_table["Model"],

    comparison_table["Recall"]

)

plt.title("Recall Comparison")

plt.xlabel("Models")

plt.ylabel("Recall")

plt.xticks(rotation=10)

plt.show()

* Recall graph banaya.

In [ ]:
# F1 Score Comperison Graph

plt.figure(figsize=(8,5))

plt.bar(

    comparison_table["Model"],

    comparison_table["F1 Score"]

)

plt.title("F1 Score Comparison")

plt.xlabel("Models")

plt.ylabel("F1 Score")

plt.xticks(rotation=10)

plt.show()

* F1 Score graph banaya.

In [ ]:
# Disply All Confusion Metrics


print("KNN Confusion Matrix")

print(cm)

print()

print("Linear SVM Confusion Matrix")

print(linear_cm)

print()

print("RBF SVM Confusion Matrix")

print(rbf_cm)

print()

print("Naive Bayes Confusion Matrix")

print(nb_cm)

* Sabhi models ki confusion matrices print ki.

In [ ]:
# Disply Classification Report

print("========== KNN ==========")

print(classification_report(

    y_test,

    y_pred

))

print("========== Linear SVM ==========")

print(classification_report(

    y_test,

    linear_predicton

))

print("========== RBF SVM ==========")

print(classification_report(

    y_test,

    rbf_prediction

))

print("========== Naive Bayes ==========")

print(classification_report(

    y_test,

    nb_prediction

))

* Sabhi models ki classification reports print ki.

In [ ]:
# Final Comperison Summary

print("=========== Model Comparison Summary ===========")

print(comparison_table)

* Comparison ka final summary print kiya.

In [ ]:
# Best Overall Model

overall_model = comparison_table.loc[

    comparison_table["Accuracy"].idxmax(),

    "Model"

]

print("Best Overall Model :", overall_model)

* Overall best model identify kiya.

Part G: Final Analysis Report 

* Markdown heading.

In [ ]:
# Disply Comperison Table

print("========== Final Model Comparison ==========")

comparison_table

* Comparison table display ki.

In [ ]:
# Find The Best Accuracy Model

best_accuracy_model = comparison_table.loc[
    comparison_table["Accuracy"].idxmax()
]

print("Best Accuracy Model")

print(best_accuracy_model)

* Accuracy ke hisaab se best model nikala.

In [ ]:
# Find Best Precision Model

best_precision_model = comparison_table.loc[
    comparison_table["Precision"].idxmax()
]

print("Best Precision Model")

print(best_precision_model)

* Precision ke hisaab se best model nikala.

In [ ]:
# Find Best Recall Model

best_recall_model = comparison_table.loc[
    comparison_table["Recall"].idxmax()
]

print("Best Recall Model")

print(best_recall_model)

* Recall ke hisaab se best model nikala.

In [ ]:
# Find The Best f1 Score Model

best_f1_model = comparison_table.loc[
    comparison_table["F1 Score"].idxmax()
]

print("Best F1 Score Model")

print(best_f1_model)

* Best F1 Score model nikala.

In [ ]:
# Select Best Model Automatically

best_model = comparison_table.loc[
    comparison_table["Accuracy"].idxmax(),
    "Model"
]

print("Best Model Selected :", best_model)

* Automatically best model select kiya.

In [ ]:
if best_model == "KNN":

    print("KNN is selected because it achieved the highest accuracy.")

elif best_model == "Linear SVM":

    print("Linear SVM is selected because it achieved the highest accuracy.")

elif best_model == "RBF SVM":

    print("RBF SVM is selected because it achieved the highest accuracy.")

else:

    print("Naive Bayes is selected because it achieved the highest accuracy.")

* Bataya ki ye model best kyun hai.

In [ ]:
# Save Comperison Table

comparison_table.to_csv(

    "Model_Comparison_Result.csv",

    index=False

)

print("Comparison table saved successfully.")

*   Comparison table ko CSV file me save kiya.